<a href="https://colab.research.google.com/github/cristiangromero/LGRT-PIA/blob/main/1er_Parcial_PIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Sistema Experto para Clasificación y Triaje de Incidentes de TI
Cátedra: Principios de Inteligencia Artificial - UGD
Paradigma: IA simbólica / Sistema basado en reglas
Mecanismo de inferencia: Encadenamiento hacia adelante (Forward Chaining)
"""


# ============================================================
# 1. REPRESENTACIÓN DE HECHOS
# ============================================================

class Hecho:
    """Representa un hecho simbólico almacenado en la memoria de trabajo."""

    def __init__(self, nombre, valor):
        self.nombre = nombre
        self.valor = valor

    def __eq__(self, otro):
        return (
            isinstance(otro, Hecho)
            and self.nombre == otro.nombre
            and self.valor == otro.valor
        )

    def __hash__(self):
        return hash((self.nombre, self.valor))

    def __repr__(self):
        return f"{self.nombre}({self.valor})"


# ============================================================
# 2. REPRESENTACIÓN DE REGLAS
# ============================================================

class Regla:
    """
    Representa una regla de producción:

    SI se cumplen todos los antecedentes
    ENTONCES se agregan los consecuentes.
    """

    def __init__(
        self,
        identificador,
        descripcion,
        antecedentes,
        consecuentes,
        prioridad=1
    ):
        self.identificador = identificador
        self.descripcion = descripcion
        self.antecedentes = antecedentes
        self.consecuentes = consecuentes
        self.prioridad = prioridad

    def puede_dispararse(self, hechos):
        """Devuelve True si todos los antecedentes están presentes."""
        return all(hecho in hechos for hecho in self.antecedentes)


# ============================================================
# 3. MEMORIA DE TRABAJO
# ============================================================

class MemoriaDeTrabajo:
    """Almacena los hechos iniciales y los hechos inferidos."""

    def __init__(self):
        self.hechos = set()
        self.traza = []

    def agregar_hecho(self, hecho, origen):
        """Agrega un hecho si todavía no existe."""
        if hecho not in self.hechos:
            self.hechos.add(hecho)
            self.traza.append(
                f"HECHO AGREGADO: {hecho} | Origen: {origen}"
            )
            return True

        return False


# ============================================================
# 4. MOTOR DE INFERENCIA
# ============================================================

class MotorDeInferencia:
    """
    Motor de inferencia mediante encadenamiento hacia adelante.

    Parte de los hechos iniciales y aplica las reglas que se
    encuentran satisfechas hasta que no pueden generarse
    nuevos hechos.
    """

    def __init__(self, reglas):
        # Las reglas con mayor prioridad se evalúan primero.
        self.reglas = sorted(
            reglas,
            key=lambda regla: regla.prioridad,
            reverse=True
        )

    def ejecutar(self, memoria):
        reglas_disparadas = set()

        memoria.traza.append(
            "\n=== INICIO DE LA INFERENCIA ==="
        )

        hubo_cambios = True

        while hubo_cambios:
            hubo_cambios = False

            for regla in self.reglas:

                # Evita disparar la misma regla más de una vez.
                if regla.identificador in reglas_disparadas:
                    continue

                if regla.puede_dispararse(memoria.hechos):

                    reglas_disparadas.add(regla.identificador)

                    memoria.traza.append(
                        f"REGLA DISPARADA: {regla.identificador} - "
                        f"{regla.descripcion}"
                    )

                    for consecuente in regla.consecuentes:
                        agregado = memoria.agregar_hecho(
                            consecuente,
                            f"Inferencia mediante {regla.identificador}"
                        )

                        if agregado:
                            hubo_cambios = True

        memoria.traza.append(
            "=== FIN DE LA INFERENCIA ==="
        )


# ============================================================
# 5. BASE DE CONOCIMIENTO
# ============================================================

def crear_base_de_conocimiento():
    """
    Crea las 15 reglas del sistema experto.
    """

    H = Hecho

    return [

        # ----------------------------------------------------
        # SEGURIDAD
        # ----------------------------------------------------

        Regla(
            "R1",
            "Detectar amenaza grave por cifrado de archivos",
            [
                H("Sintoma", "Cifrado_Anomalo_Archivos")
            ],
            [
                H("Tipo", "Seguridad"),
                H("IndicadorAmenaza", "Grave")
            ],
            prioridad=10
        ),

        Regla(
            "R2",
            "Asignar P1 y equipo de Seguridad a una amenaza grave",
            [
                H("Tipo", "Seguridad"),
                H("IndicadorAmenaza", "Grave")
            ],
            [
                H("Prioridad", "P1"),
                H("Equipo", "SecOps")
            ],
            prioridad=10
        ),

        # ----------------------------------------------------
        # REDES
        # ----------------------------------------------------

        Regla(
            "R3",
            "Detectar falla crítica de red",
            [
                H("ServicioAfectado", "Enrutador_Core_Red"),
                H("Sintoma", "Sin_Conectividad_Red")
            ],
            [
                H("Tipo", "Redes"),
                H("Impacto", "Alto")
            ],
            prioridad=8
        ),

        Regla(
            "R4",
            "Asignar P1 y NetOps a una falla de red global",
            [
                H("Tipo", "Redes"),
                H("Impacto", "Alto"),
                H("Alcance", "Global")
            ],
            [
                H("Prioridad", "P1"),
                H("Equipo", "NetOps")
            ],
            prioridad=8
        ),

        # ----------------------------------------------------
        # INFRAESTRUCTURA
        # ----------------------------------------------------

        Regla(
            "R5",
            "Detectar falla severa de hardware",
            [
                H("ServicioAfectado", "Base_De_Datos"),
                H("Sintoma", "Falla_Fisica_Disco")
            ],
            [
                H("Tipo", "Infraestructura"),
                H("SeveridadHardware", "Alta")
            ],
            prioridad=7
        ),

        Regla(
            "R6",
            "Asignar P2 y SysAdmin a una falla de hardware departamental",
            [
                H("Tipo", "Infraestructura"),
                H("SeveridadHardware", "Alta"),
                H("Alcance", "Departamental")
            ],
            [
                H("Prioridad", "P2"),
                H("Equipo", "SysAdmin")
            ],
            prioridad=7
        ),

        # ----------------------------------------------------
        # DESARROLLO / APLICACIONES
        # ----------------------------------------------------

        Regla(
            "R7",
            "Detectar error crítico en una aplicación ERP",
            [
                H("ServicioAfectado", "Core_ERP"),
                H("Sintoma", "Error_Codigo_500_App")
            ],
            [
                H("Tipo", "Desarrollo_Aplicaciones"),
                H("FalloSoftware", "Critico")
            ],
            prioridad=6
        ),

        Regla(
            "R8",
            "Asignar P1 y AppSupport a una falla global del ERP",
            [
                H("Tipo", "Desarrollo_Aplicaciones"),
                H("FalloSoftware", "Critico"),
                H("Alcance", "Global")
            ],
            [
                H("Prioridad", "P1"),
                H("Equipo", "AppSupport")
            ],
            prioridad=6
        ),

        Regla(
            "R9",
            "Asignar P3 y AppSupport a una falla individual del ERP",
            [
                H("Tipo", "Desarrollo_Aplicaciones"),
                H("FalloSoftware", "Critico"),
                H("Alcance", "Individual")
            ],
            [
                H("Prioridad", "P3"),
                H("Equipo", "AppSupport")
            ],
            prioridad=5
        ),

        # ----------------------------------------------------
        # PROBLEMAS DE RED INDIVIDUALES
        # ----------------------------------------------------

        Regla(
            "R10",
            "Asignar P3 a un problema de red individual",
            [
                H("Tipo", "Redes"),
                H("Alcance", "Individual")
            ],
            [
                H("Prioridad", "P3"),
                H("Equipo", "NetOps")
            ],
            prioridad=4
        ),

        # ----------------------------------------------------
        # ESTACIONES DE TRABAJO
        # ----------------------------------------------------

        Regla(
            "R11",
            "Clasificar una estación con lentitud general como impacto bajo",
            [
                H("ServicioAfectado", "Estacion_Trabajo"),
                H("Sintoma", "Lentitud_General")
            ],
            [
                H("Tipo", "Infraestructura"),
                H("Impacto", "Bajo")
            ],
            prioridad=3
        ),

        Regla(
            "R12",
            "Asignar P4 y Service Desk a un incidente individual de bajo impacto",
            [
                H("Tipo", "Infraestructura"),
                H("Impacto", "Bajo"),
                H("Alcance", "Individual")
            ],
            [
                H("Prioridad", "P4"),
                H("Equipo", "ServiceDesk_Tier1")
            ],
            prioridad=3
        ),

        # ----------------------------------------------------
        # ESCALAMIENTO
        # ----------------------------------------------------

        Regla(
            "R13",
            "Detectar nivel crítico extremo",
            [
                H("Impacto", "Alto"),
                H("Urgencia", "Alta")
            ],
            [
                H("NivelCritico", "Extremo")
            ],
            prioridad=9
        ),

        Regla(
            "R14",
            "Asignar P1 cuando el nivel crítico es extremo",
            [
                H("NivelCritico", "Extremo")
            ],
            [
                H("Prioridad", "P1")
            ],
            prioridad=9
        ),

        # ----------------------------------------------------
        # SEGURIDAD NO GRAVE
        # ----------------------------------------------------

        Regla(
            "R15",
            "Asignar P2 y Seguridad a un incidente de seguridad no grave",
            [
                H("Tipo", "Seguridad"),
                H("IndicadorAmenaza", "NoGrave")
            ],
            [
                H("Prioridad", "P2"),
                H("Equipo", "SecOps")
            ],
            prioridad=5
        )
    ]


# ============================================================
# 6. OBTENCIÓN DEL RESULTADO
# ============================================================

def obtener_resultado(memoria):
    """
    Obtiene el resultado final del sistema.

    Si se infieren varias prioridades, se conserva la de mayor
    criticidad: P1 > P2 > P3 > P4.
    """

    prioridades = [
        hecho.valor
        for hecho in memoria.hechos
        if hecho.nombre == "Prioridad"
    ]

    tipos = [
        hecho.valor
        for hecho in memoria.hechos
        if hecho.nombre == "Tipo"
    ]

    equipos = [
        hecho.valor
        for hecho in memoria.hechos
        if hecho.nombre == "Equipo"
    ]

    if prioridades:
        prioridad_final = sorted(
            prioridades,
            key=lambda prioridad: int(prioridad[1:])
        )[0]
    else:
        prioridad_final = "No determinada"

    return {
        "Prioridad": prioridad_final,
        "Tipo": tipos[0] if tipos else "No determinado",
        "Equipo": equipos[0] if equipos else "No determinado"
    }


# ============================================================
# 7. EJECUCIÓN DE UNA PRUEBA
# ============================================================

def ejecutar_prueba():
    print("=" * 65)
    print("SISTEMA EXPERTO DE TRIAJE DE INCIDENTES DE TI")
    print("=" * 65)

    # Crear base de conocimiento
    base_conocimiento = crear_base_de_conocimiento()

    # Crear motor de inferencia
    motor = MotorDeInferencia(base_conocimiento)

    # Crear memoria de trabajo
    memoria = MemoriaDeTrabajo()

    # --------------------------------------------------------
    # CASO DE PRUEBA
    # Incidente de seguridad por cifrado de archivos
    # --------------------------------------------------------

    hechos_iniciales = [
        Hecho("ServicioAfectado", "Core_ERP"),
        Hecho("Sintoma", "Cifrado_Anomalo_Archivos"),
        Hecho("Alcance", "Global"),
        Hecho("Urgencia", "Alta")
    ]

    print("\nDATOS DEL INCIDENTE")
    print("-" * 65)

    for hecho in hechos_iniciales:
        print(f"- {hecho}")
        memoria.agregar_hecho(
            hecho,
            "Datos iniciales del ticket"
        )

    # Ejecutar inferencia
    motor.ejecutar(memoria)

    # Obtener resultado
    resultado = obtener_resultado(memoria)

    print("\nRESULTADO DEL TRIAJE")
    print("-" * 65)
    print(f"Prioridad : {resultado['Prioridad']}")
    print(f"Tipo      : {resultado['Tipo']}")
    print(f"Equipo    : {resultado['Equipo']}")

    # Mostrar trazabilidad
    print("\nTRAZA DE INFERENCIA")
    print("-" * 65)

    for paso in memoria.traza:
        print(paso)


# ============================================================
# 8. EJECUTAR
# ============================================================

if __name__ == "__main__":
    ejecutar_prueba()